In [1]:
import sys
import os
from pathlib import Path
from sqlalchemy import select

sys.path.append(os.path.abspath(os.path.join('..')))

from src.database.connector import db_connector
from src.database.structure import ShopeeOrder, ComboVariant, Combo, Variant, ComboDetail
from sqlalchemy import select, func, join, outerjoin

In [2]:
BASE_PATH = Path().cwd().parent
DB_PATH = Path(BASE_PATH / "database.sqlite3")

session = db_connector(DB_PATH)

In [3]:
# Lấy các phiên bản của combo variant đã map trong bảng combo_details
get_cv_mapped = (
        select(
            ComboVariant.combo_key,
            ComboVariant.variant_key,
            func.sum(ComboDetail.product_price * ComboDetail.product_quantity).label("total_combo_value")
        )
        .join(ComboVariant, ComboDetail.combo_variant_key == ComboVariant.combo_variant_key)
        .group_by(ComboVariant.combo_key, ComboVariant.variant_key, ComboDetail.combo_composition_key)
    )
cv_mapped_versions = session.execute(get_cv_mapped).all()

# Dict này dùng để so sánh nhanh giá với dữ liệu trong đơn hàng
# Cấu trúc {(combo_key, variant_key): total_combo_price),...}
cv_compare_list = {}
for version in cv_mapped_versions:
    cv_key_pair = (version.combo_key, version.variant_key)
    if cv_key_pair not in cv_compare_list: # Nếu cặp đang được lặp qua chưa tồn tại trong dict thì gán nó làm key
        cv_compare_list[cv_key_pair] = set() # Gán cho value của key hiện tại là một set
    cv_compare_list[cv_key_pair].add(version.total_combo_value) # Nếu đã tồn tại thì gán tổng giá trị của version đó vào set (value của dict)

# Lấy ra giá trị lớn nhất đang gán cho combo_composition_key trong database
composition_key_available = session.scalar(select(func.max(ComboDetail.combo_composition_key)))

# Khung list các dict chứa thông tin được import vào db
cv_import_cache = []

# Lấy các phiên bản của combo variant nằm trong bảng shopee_order
get_cv_order = select(
        ShopeeOrder.combo_key,
        ShopeeOrder.variant_key,
        ShopeeOrder.combo_name,
        ShopeeOrder.variant_name,
        ShopeeOrder.deal_price
    ).distinct()
cv_order_versions = session.execute(get_cv_order).all()

for order in cv_order_versions:
    cv_key_pair = (order.combo_key, order.variant_key)
    order_price = order.deal_price

    # Flag đánh dấu version thiếu
    is_missing = (cv_key_pair not in cv_compare_list) or (order_price not in cv_compare_list[cv_key_pair])

    if is_missing:
        composition_key_available +=1
        cv_key = session.scalar(
            select(ComboVariant.combo_variant_key)
            .where(ComboVariant.combo_key == order.combo_key)
            .where(ComboVariant.variant_key == order.variant_key)
        )

        combo_template = {
            "combo_key": order.combo_key,
            "variant_key": order.variant_key,
            "combo_name": order.combo_name,
            "variant_name": order.variant_name if order.variant_name else "-",
            "deal_price": float(order_price),
            "combo_variant_key": cv_key,
            "combo_composition_key": composition_key_available,
            "products": [
                {
                    "product_key": None,
                    "product_code": "",
                    "product_name": "",
                    "product_quantity": 0,
                    "product_price": 0.0
                }
            ]
        }
        cv_import_cache.append(combo_template)

In [4]:
cv_import_cache

[{'combo_key': 62,
  'variant_key': None,
  'combo_name': "Rong biển / lá kim cuốn cơm Hàn Quốc O'food 10g, sử dụng cho các món kimbap, sushi",
  'variant_name': '-',
  'deal_price': 28000.0,
  'combo_variant_key': 107,
  'combo_composition_key': 4,
  'products': [{'product_key': None,
    'product_code': '',
    'product_name': '',
    'product_quantity': 0,
    'product_price': 0.0}]},
 {'combo_key': 60,
  'variant_key': None,
  'combo_name': "[Mua 10 tặng 6] Lốc 16 gói lá kim tẩm dầu oliu 5g O'food",
  'variant_name': '-',
  'deal_price': 135000.0,
  'combo_variant_key': 100,
  'combo_composition_key': 5,
  'products': [{'product_key': None,
    'product_code': '',
    'product_name': '',
    'product_quantity': 0,
    'product_price': 0.0}]},
 {'combo_key': 23,
  'variant_key': None,
  'combo_name': '[PHIÊN BẢN 24 GÓI] Rong Biển Lá Kim O’Food Ô Liu 4G – Combo 12+12 – Ăn Liền Healthy',
  'variant_name': '-',
  'deal_price': 152000.0,
  'combo_variant_key': 29,
  'combo_composition_k